In [ ]:
import os
import findspark
import kagglehub

findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better spark

In [3]:
# Download latest version
path = kagglehub.dataset_download("jakewright/9000-tickers-of-stock-market-data-full-history")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\hille\.cache\kagglehub\datasets\jakewright\9000-tickers-of-stock-market-data-full-history\versions\2


In [4]:
path = path + "/all_stock_data.csv"
print(path)

C:\Users\hille\.cache\kagglehub\datasets\jakewright\9000-tickers-of-stock-market-data-full-history\versions\2/all_stock_data.csv


In [5]:
from pyspark.sql.types import *

# Definimos el esquema del data set
schema = StructType([

    StructField("Date", DateType(), True),
    StructField("Ticker", StringType(), True),

    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),

    StructField("Volume", DoubleType(), True),
    StructField("Dividends", DecimalType(5,1), True),
    StructField("Stock Splits", DecimalType(5,1), True)

])

In [6]:
# Disparador 0
# Leemos el data set e imprimimos los primeors 5 registros

df = spark.read.csv(path, header=True, inferSchema=False, schema=schema)
df.show(5)

+----------+------+----+-------------------+-------------------+-------------------+---------+---------+------------+
|      Date|Ticker|Open|               High|                Low|              Close|   Volume|Dividends|Stock Splits|
+----------+------+----+-------------------+-------------------+-------------------+---------+---------+------------+
|1962-01-02|    ED| 0.0| 0.2658275556233194|0.26178762316703796|0.26178762316703796|  25600.0|      0.0|         0.0|
|1962-01-02|   CVX| 0.0|0.04680890217423439|0.04606926600933256|0.04680890217423439| 105840.0|      0.0|         0.0|
|1962-01-02|    GD| 0.0|0.21003275954390174|0.20306070787008793| 0.2082897424697876|2648000.0|      0.0|         0.0|
|1962-01-02|    BP| 0.0|0.14143933090345925|0.13952797651290894|0.13952797651290894|  77440.0|      0.0|         0.0|
|1962-01-02|   MSI| 0.0| 0.7649229763450202| 0.7452535214492476| 0.7518101930618286|  65671.0|      0.0|         0.0|
+----------+------+----+-------------------+------------

In [7]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Ticker: string (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Dividends: decimal(5,1) (nullable = true)
 |-- Stock Splits: decimal(5,1) (nullable = true)



In [ ]:
from pyspark.sql.functions import count, when, col

# Definimos el cache
# df.cache() Esto hace que se guarde en la RAM, comentarlo hace que se lea desde el disco.

resumen = df.describe()

valores_nulos = df.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
])

In [8]:
# Disparador 1
num = df.count()

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "C:\Users\hille\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hille\AppData\Local\Temp\ipykernel_20696\3881292400.py", line 2, in <module>
    num = df.count()
  File "c:\Users\hille\anaconda3\envs\env_pyspark\Lib\site-packages\pyspark\sql\classic\dataframe.py", line 439, in count
    return int(self._jdf.count())
               ~~~~~~~~~~~~~~~^^
  File "c:\Users\hille\anaconda3\envs\env_pyspark\Lib\site-packages\py4j\java_gateway.py", line 1362, in __call__
    return_value = get_return_value(
        answer, self.gateway_client, self.target_id, self.name)
  File "c:\Users\hille\anaconda3\envs\env_pyspark\Lib\site-packages\pyspark\errors\exceptions\captured.py", line 263, in deco
    return f(*a, **kw)
  File "c:\Users\

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
# Imprimimos el numero de columnas y registros
print(f"Número de columnas: {len(df.columns)}")
print(f"Número de registros: {num:,}\n")

Número de columnas: 9


NameError: name 'num' is not defined

In [ ]:
# Disparador 2
resumen_transpuesto = resumen.toPandas().set_index('summary').T

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
# Imprimimos el  resumen
resumen_transpuesto

NameError: name 'resumen_transpuesto' is not defined

In [ ]:
# Disparador 3
nulos_pandas = valores_nulos.toPandas().T

In [ ]:
# Imprimimos los valores nulos
nulos_pandas.columns = ['Valores nulos']
nulos_pandas["%"] = (nulos_pandas["Valores nulos"] / num) * 100

nulos_pandas


,Valores nulos,%
Date,0,0.000000
Ticker,0,0.000000
Open,109,0.000315
High,109,0.000315
Low,109,0.000315
Close,106,0.000306
Volume,0,0.000000
Dividends,0,0.000000
Stock Splits,3,0.000009


# Caracterización de la población y selección de variables

Como parte inicial del proyecto, se realizó la caracterización de la población en un documento complementario, donde se analizaron las principales características del dataset, incluyendo sus variables, tipos de datos, estadísticas descriptivas y posibles inconsistencias en los datos.

Con base en este análisis, se seleccionaron las variables **Ticker**, **Date** y **Volume** para realizar el particionamiento de la población, ya que permiten segmentar los datos por acción bursátil, periodo de tiempo y nivel de actividad en el mercado.

Estas variables servirán como base para organizar la población y facilitar la aplicación de técnicas de muestreo en las siguientes etapas del proyecto.